# EGN 321 — Module 7 Assignment 7.1 Complete Demo
## Unattended System: Monitor, Alert, Log, Recover, and Prove It

This version is aligned directly to the Assignment 7.1 required evidence and deliverables.

### Required evidence created by this notebook
1. Dashboard running
2. Current reading
3. Historical view
4. Alert active
5. Alert cleared
6. Stale or missing-data condition
7. Logs
8. Restart/recovery
9. Final healthy state

### Deliverables created
- Flask application source code
- SQLite database
- alert logic
- logging configuration
- startup/restart configuration
- dashboard evidence
- alert evidence
- application log
- recovery-test evidence
- complete README
- `AI_LOG.md`
- GitHub submission checklist
- final ZIP package

> Colab is used as the teaching/demo environment. The generated Flask project should be run locally or on an approved host for the student's real unattended deployment.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3, json, math, random, zipfile, shutil, subprocess, sys
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone

BASE = Path("/content/module7_assignment7_1_complete_demo")
BASE.mkdir(exist_ok=True)

DATA_FILE = BASE / "sensor_replay.csv"
DB_FILE = BASE / "module7_monitor.db"
LOG_FILE = BASE / "module7_app.jsonl"
APP_DIR = BASE / "flask_project"
EXPORT = BASE / "assignment7_1_submission"
EVIDENCE = EXPORT / "required_evidence"

APP_DIR.mkdir(exist_ok=True)
EVIDENCE.mkdir(parents=True, exist_ok=True)

print("Working directory:", BASE)


## Step 1 — Create the virtual sensor replay

The demo data contains:
- normal variation;
- a sustained threshold event;
- missing readings;
- rejected readings;
- recovery to normal conditions.


In [ ]:
random.seed(71)
rows = []
start = datetime(2026,10,13,18,0,0,tzinfo=timezone.utc)

for i in range(1,241):
    ts = start + timedelta(seconds=(i-1)*2)
    temp = 23.0 + 0.35*math.sin(i/12) + random.gauss(0,0.12)
    status = "OK"
    reason = ""

    if 90 <= i <= 110:
        temp = 30.4 + 0.25*math.sin(i/3) + random.gauss(0,0.08)

    if 145 <= i <= 150:
        temp = np.nan
        status = "MISSING"
        reason = "No sensor value received"

    if i in [175,176]:
        temp = 67.0 if i == 175 else -8.0
        status = "REJECTED"
        reason = "Outside supported classroom range 0-50 C"

    rows.append({
        "sample_number": i,
        "timestamp": ts.isoformat(),
        "temperature_c": temp,
        "sensor_status": status,
        "status_reason": reason
    })

sensor = pd.DataFrame(rows)
sensor.to_csv(DATA_FILE,index=False)

display(sensor.head())
display(sensor["sensor_status"].value_counts().rename_axis("status").reset_index(name="count"))


## Step 2 — Deterministic alert, health, and logging logic


In [ ]:
class AlertEngine:
    def __init__(self, trigger=30.0, clear=28.5, consecutive=3):
        self.trigger = trigger
        self.clear = clear
        self.consecutive = consecutive
        self.high_count = 0
        self.state = "NORMAL"

    def update(self, value, valid=True):
        previous = self.state

        if not valid or value is None:
            self.high_count = 0
            return self.state, previous != self.state

        if self.state == "NORMAL":
            if value >= self.trigger:
                self.high_count += 1
                if self.high_count >= self.consecutive:
                    self.state = "ALERT"
            else:
                self.high_count = 0

        elif self.state == "ALERT":
            if value <= self.clear:
                self.state = "NORMAL"
                self.high_count = 0

        return self.state, previous != self.state


def health_state(sensor_status, seconds_since_last_valid):
    if sensor_status in {"MISSING","REJECTED"}:
        return "DEGRADED"
    if seconds_since_last_valid is None:
        return "FAILED"
    if seconds_since_last_valid > 30:
        return "FAILED"
    if seconds_since_last_valid > 10:
        return "DEGRADED"
    return "HEALTHY"


def log_event(level, event, **fields):
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "level": level,
        "event": event,
        **fields
    }
    with open(LOG_FILE,"a",encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")
    return record


## Step 3 — Build SQLite history and operational evidence


In [ ]:
for p in [DB_FILE,LOG_FILE]:
    if p.exists():
        p.unlink()

conn = sqlite3.connect(DB_FILE)
conn.execute(
    "CREATE TABLE readings("
    "id INTEGER PRIMARY KEY AUTOINCREMENT,"
    "sample_number INTEGER,"
    "timestamp TEXT,"
    "raw_value REAL,"
    "accepted_value REAL,"
    "sensor_status TEXT,"
    "health_state TEXT,"
    "alert_state TEXT)"
)

engine = AlertEngine()
last_valid_ts = None

for _, r in sensor.iterrows():
    ts = pd.to_datetime(r["timestamp"],utc=True)
    raw = r["temperature_c"]
    status = str(r["sensor_status"]).upper()

    valid = (status == "OK" and pd.notna(raw))
    accepted = float(raw) if valid else None

    if valid:
        last_valid_ts = ts

    age = None if last_valid_ts is None else (ts-last_valid_ts).total_seconds()
    h = health_state(status, age)
    alert, changed = engine.update(accepted, valid)

    conn.execute(
        "INSERT INTO readings "
        "(sample_number,timestamp,raw_value,accepted_value,sensor_status,health_state,alert_state) "
        "VALUES(?,?,?,?,?,?,?)",
        (
            int(r["sample_number"]), r["timestamp"],
            None if pd.isna(raw) else float(raw),
            accepted, status, h, alert
        )
    )

    log_event(
        "INFO" if valid else "WARNING",
        "reading_processed",
        sample_number=int(r["sample_number"]),
        sensor_status=status,
        health_state=h,
        alert_state=alert
    )

    if changed:
        log_event(
            "ALERT" if alert == "ALERT" else "INFO",
            "alert_activated" if alert == "ALERT" else "alert_cleared",
            sample_number=int(r["sample_number"]),
            value=accepted,
            alert_state=alert
        )

conn.commit()
history = pd.read_sql_query("SELECT * FROM readings ORDER BY id",conn)
conn.close()

display(history.head())


## Step 4 — Generate the Flask application and restart configuration


In [ ]:
app_py = [
"from flask import Flask, jsonify",
"import sqlite3, os",
"",
'DB_PATH = os.environ.get("MODULE7_DB","module7_monitor.db")',
"app = Flask(__name__)",
"",
'@app.get("/")',
"def dashboard():",
"    conn = sqlite3.connect(DB_PATH)",
'    latest = conn.execute("SELECT sample_number,timestamp,accepted_value,sensor_status,health_state,alert_state FROM readings ORDER BY id DESC LIMIT 1").fetchone()',
'    count = conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]',
'    recent = conn.execute("SELECT sample_number,timestamp,accepted_value,sensor_status,health_state,alert_state FROM readings ORDER BY id DESC LIMIT 20").fetchall()',
"    conn.close()",
'    return {"latest":latest,"history_count":count,"recent":recent}',
"",
'@app.get("/health")',
"def health():",
"    conn = sqlite3.connect(DB_PATH)",
'    row = conn.execute("SELECT health_state FROM readings ORDER BY id DESC LIMIT 1").fetchone()',
"    conn.close()",
'    return jsonify({"health_state": row[0] if row else "UNKNOWN"})',
"",
'if __name__ == "__main__":',
'    app.run(host="0.0.0.0",port=5000)'
]
(APP_DIR/"app.py").write_text("\n".join(app_py),encoding="utf-8")

supervisor_py = [
"import subprocess, sys, time",
"while True:",
'    print("SUPERVISOR: starting app")',
'    p = subprocess.Popen([sys.executable,"app.py"])',
"    code = p.wait()",
'    print("SUPERVISOR: app exited",code)',
"    time.sleep(1)"
]
(APP_DIR/"supervisor.py").write_text("\n".join(supervisor_py),encoding="utf-8")

alert_logic_py = [
"class AlertEngine:",
"    def __init__(self, trigger=30.0, clear=28.5, consecutive=3):",
"        self.trigger=trigger",
"        self.clear=clear",
"        self.consecutive=consecutive",
"        self.high_count=0",
'        self.state="NORMAL"',
"",
"    def update(self,value,valid=True):",
"        previous=self.state",
"        if not valid or value is None:",
"            self.high_count=0",
"            return self.state, previous != self.state",
'        if self.state=="NORMAL":',
"            if value >= self.trigger:",
"                self.high_count += 1",
"                if self.high_count >= self.consecutive:",
'                    self.state="ALERT"',
"            else:",
"                self.high_count=0",
'        elif self.state=="ALERT" and value <= self.clear:',
'            self.state="NORMAL"',
"            self.high_count=0",
"        return self.state, previous != self.state"
]
(APP_DIR/"alert_logic.py").write_text("\n".join(alert_logic_py),encoding="utf-8")

logging_config_py = [
"import json",
"from datetime import datetime, timezone",
"",
"def write_log(path, level, event, **fields):",
'    record={"timestamp":datetime.now(timezone.utc).isoformat(),"level":level,"event":event,**fields}',
'    with open(path,"a",encoding="utf-8") as f:',
'        f.write(json.dumps(record)+"\\n")',
"    return record"
]
(APP_DIR/"logging_config.py").write_text("\n".join(logging_config_py),encoding="utf-8")

(APP_DIR/"requirements.txt").write_text("flask\npandas\n",encoding="utf-8")

systemd_lines = [
"[Unit]",
"Description=EGN321 Module 7 Flask Monitor",
"After=network.target",
"",
"[Service]",
"Type=simple",
"WorkingDirectory=/path/to/flask_project",
"ExecStart=/path/to/python app.py",
"Restart=always",
"RestartSec=3",
"Environment=MODULE7_DB=/path/to/module7_monitor.db",
"",
"[Install]",
"WantedBy=multi-user.target"
]
(APP_DIR/"module7-monitor.service.example").write_text("\n".join(systemd_lines),encoding="utf-8")

print("Generated Flask/source files:")
for p in sorted(APP_DIR.iterdir()):
    print(" -",p.name)


## Step 5 — Verify the dashboard


In [ ]:
import importlib.util, os

os.environ["MODULE7_DB"] = str(DB_FILE)

spec = importlib.util.spec_from_file_location("module7_app",APP_DIR/"app.py")
appmod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(appmod)

client = appmod.app.test_client()
resp = client.get("/")
health_resp = client.get("/health")

dashboard_json = resp.get_json()

print("Dashboard HTTP status:",resp.status_code)
print(json.dumps(dashboard_json,indent=2)[:2000])
print("Health endpoint:",health_resp.get_json())


## Step 6 — Controlled crash and restart/recovery


In [ ]:
worker = BASE/"demo_worker.py"
flag = BASE/"worker_failed_once.flag"

if flag.exists():
    flag.unlink()

worker_lines = [
"from pathlib import Path",
"import sys, time",
'flag=Path("worker_failed_once.flag")',
"if not flag.exists():",
'    flag.write_text("yes")',
'    print("WORKER: deliberate crash")',
"    sys.exit(7)",
'print("WORKER: restarted successfully")',
"time.sleep(0.5)",
'print("WORKER: healthy")',
"sys.exit(0)"
]
worker.write_text("\n".join(worker_lines),encoding="utf-8")

attempts = []

for attempt in range(1,4):
    p = subprocess.run(
        [sys.executable,worker.name],
        cwd=str(BASE),
        capture_output=True,
        text=True
    )

    attempts.append({
        "attempt":attempt,
        "returncode":p.returncode,
        "stdout":p.stdout.strip()
    })

    log_event(
        "ERROR" if p.returncode else "INFO",
        "supervisor_attempt",
        attempt=attempt,
        returncode=p.returncode,
        output=p.stdout.strip()
    )

    if p.returncode == 0:
        break

recovery = pd.DataFrame(attempts)

assert recovery.iloc[0]["returncode"] != 0
assert recovery.iloc[-1]["returncode"] == 0

display(recovery)
print("✅ Controlled recovery test passed.")


# Required Evidence

The following cells create every evidence item named in the assignment.


In [ ]:
records = []
with open(LOG_FILE,"r",encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
logs = pd.DataFrame(records)

latest = dashboard_json["latest"]

dashboard_evidence = {
    "http_status": int(resp.status_code),
    "dashboard_response": dashboard_json,
    "health_endpoint": health_resp.get_json()
}
(EVIDENCE/"dashboard_running.json").write_text(json.dumps(dashboard_evidence,indent=2),encoding="utf-8")

html_lines = [
"<!DOCTYPE html>",
"<html><head><meta charset='utf-8'><title>Module 7 Dashboard Evidence</title></head><body>",
"<h1>EGN 321 Module 7 — Unattended Dashboard</h1>",
f"<h2>Current Reading: {latest[2]} °C</h2>",
f"<p>Sample: {latest[0]}</p>",
f"<p>Timestamp: {latest[1]}</p>",
f"<p>Sensor status: {latest[3]}</p>",
f"<p>System health: {latest[4]}</p>",
f"<p>Alert state: {latest[5]}</p>",
f"<p>History rows: {dashboard_json['history_count']}</p>",
"</body></html>"
]
(EVIDENCE/"dashboard_snapshot.html").write_text("\n".join(html_lines),encoding="utf-8")

plt.figure(figsize=(10,5))
plt.axis("off")
plt.text(.03,.88,"EGN 321 — Module 7 Dashboard",fontsize=19,weight="bold")
plt.text(.03,.69,f"Current reading: {latest[2]} °C",fontsize=16)
plt.text(.03,.56,f"Sensor status: {latest[3]}",fontsize=13)
plt.text(.03,.43,f"System health: {latest[4]}",fontsize=13)
plt.text(.03,.30,f"Alert state: {latest[5]}",fontsize=13)
plt.text(.03,.17,f"History rows: {dashboard_json['history_count']}",fontsize=13)
plt.text(.03,.05,f"HTTP status: {resp.status_code}",fontsize=11)
plt.tight_layout()
plt.savefig(EVIDENCE/"dashboard_running.png",dpi=160,bbox_inches="tight")
plt.close()

print("✅ Evidence 1: dashboard running")


In [ ]:
current = history.iloc[[-1]][
    ["sample_number","timestamp","accepted_value","sensor_status","health_state","alert_state"]
].copy()

current.to_csv(EVIDENCE/"current_reading.csv",index=False)

print("✅ Evidence 2: current reading")
display(current)


In [ ]:
history.to_csv(EVIDENCE/"historical_view.csv",index=False)

plt.figure(figsize=(12,5))
plt.plot(history["sample_number"],history["accepted_value"],linewidth=1.2)
plt.xlabel("Sample Number")
plt.ylabel("Accepted Temperature (°C)")
plt.title("Historical Sensor View")
plt.grid(True,alpha=.25)
plt.tight_layout()
plt.savefig(EVIDENCE/"historical_view.png",dpi=160)
plt.close()

print("✅ Evidence 3: historical view")


In [ ]:
alert_active = history[history["alert_state"].eq("ALERT")].copy()

if alert_active.empty:
    raise RuntimeError("No alert-active evidence found.")

alert_active.to_csv(EVIDENCE/"alert_active_rows.csv",index=False)

r = alert_active.iloc[0]

plt.figure(figsize=(9,4))
plt.axis("off")
plt.text(.04,.78,"ALERT ACTIVE",fontsize=20,weight="bold")
plt.text(.04,.58,f"Sample: {int(r['sample_number'])}",fontsize=14)
plt.text(.04,.43,f"Value: {r['accepted_value']:.2f} °C",fontsize=14)
plt.text(.04,.28,f"Alert: {r['alert_state']}",fontsize=14)
plt.text(.04,.13,f"Health: {r['health_state']}",fontsize=14)
plt.tight_layout()
plt.savefig(EVIDENCE/"alert_active.png",dpi=160,bbox_inches="tight")
plt.close()

print("✅ Evidence 4: alert active")


In [ ]:
alert_indices = history.index[history["alert_state"].eq("ALERT")].tolist()
last_alert_index = max(alert_indices)

cleared = history.loc[last_alert_index+1:]
cleared = cleared[cleared["alert_state"].eq("NORMAL")].iloc[[0]].copy()
cleared.to_csv(EVIDENCE/"alert_cleared_row.csv",index=False)

r = cleared.iloc[0]

plt.figure(figsize=(9,4))
plt.axis("off")
plt.text(.04,.78,"ALERT CLEARED",fontsize=20,weight="bold")
plt.text(.04,.58,f"Sample: {int(r['sample_number'])}",fontsize=14)
plt.text(.04,.43,f"Value: {r['accepted_value']:.2f} °C",fontsize=14)
plt.text(.04,.28,f"Alert: {r['alert_state']}",fontsize=14)
plt.text(.04,.13,f"Health: {r['health_state']}",fontsize=14)
plt.tight_layout()
plt.savefig(EVIDENCE/"alert_cleared.png",dpi=160,bbox_inches="tight")
plt.close()

print("✅ Evidence 5: alert cleared")


In [ ]:
missing = history[history["sensor_status"].isin(["MISSING","REJECTED"])].copy()

missing.to_csv(EVIDENCE/"missing_or_degraded_condition.csv",index=False)

r = missing.iloc[0]

plt.figure(figsize=(9,4))
plt.axis("off")
plt.text(.04,.78,"MISSING / DEGRADED",fontsize=20,weight="bold")
plt.text(.04,.58,f"Sample: {int(r['sample_number'])}",fontsize=14)
plt.text(.04,.43,f"Sensor status: {r['sensor_status']}",fontsize=14)
plt.text(.04,.28,f"Health: {r['health_state']}",fontsize=14)
plt.text(.04,.13,"Missing evidence is preserved; it is not silently replaced.",fontsize=11)
plt.tight_layout()
plt.savefig(EVIDENCE/"missing_or_degraded.png",dpi=160,bbox_inches="tight")
plt.close()

print("✅ Evidence 6: missing/degraded condition")


In [ ]:
logs.to_csv(EVIDENCE/"application_logs.csv",index=False)
shutil.copy2(LOG_FILE,EVIDENCE/"application_logs.jsonl")

print("✅ Evidence 7: application logs")
display(logs["event"].value_counts().rename_axis("event").reset_index(name="count"))


In [ ]:
recovery.to_csv(EVIDENCE/"restart_recovery.csv",index=False)

plt.figure(figsize=(9,4))
plt.axis("off")
plt.text(.04,.82,"RESTART / RECOVERY TEST",fontsize=19,weight="bold")

for idx,row in recovery.iterrows():
    y = .64-(idx*.18)
    result = "SUCCESS" if row["returncode"] == 0 else "CRASHED"
    plt.text(.06,y,f"Attempt {int(row['attempt'])}: {result} (code {int(row['returncode'])})",fontsize=13)

plt.text(.04,.08,"Final result: controlled automatic restart succeeded.",fontsize=11)
plt.tight_layout()
plt.savefig(EVIDENCE/"restart_recovery.png",dpi=160,bbox_inches="tight")
plt.close()

print("✅ Evidence 8: restart/recovery")


In [ ]:
final_state = {
    "sensor_final_health_state": str(history.iloc[-1]["health_state"]),
    "sensor_final_alert_state": str(history.iloc[-1]["alert_state"]),
    "restart_recovery_passed": bool(recovery.iloc[-1]["returncode"] == 0),
    "database_history_rows_preserved": int(len(history))
}

final_state["final_system_healthy"] = bool(
    final_state["sensor_final_health_state"] == "HEALTHY"
    and final_state["sensor_final_alert_state"] == "NORMAL"
    and final_state["restart_recovery_passed"]
)

assert final_state["final_system_healthy"]

(EVIDENCE/"final_healthy_state.json").write_text(
    json.dumps(final_state,indent=2),
    encoding="utf-8"
)

plt.figure(figsize=(9,4))
plt.axis("off")
plt.text(.04,.80,"FINAL HEALTHY STATE",fontsize=20,weight="bold")
plt.text(.04,.58,f"Health: {final_state['sensor_final_health_state']}",fontsize=15)
plt.text(.04,.41,f"Alert: {final_state['sensor_final_alert_state']}",fontsize=15)
plt.text(.04,.24,f"Recovery passed: {final_state['restart_recovery_passed']}",fontsize=15)
plt.text(.04,.09,f"History preserved: {final_state['database_history_rows_preserved']} rows",fontsize=12)
plt.tight_layout()
plt.savefig(EVIDENCE/"final_healthy_state.png",dpi=160,bbox_inches="tight")
plt.close()

print("✅ Evidence 9: final healthy state")
print(json.dumps(final_state,indent=2))


# Complete README and Submission Documents


In [ ]:
README_TEXT = '''# EGN 321 — Module 7 Assignment 7.1
## Unattended Monitoring System

### How to Start the System
Install dependencies with:
`pip install -r requirements.txt`

Run directly:
`python app.py`

Run through the restart supervisor:
`python supervisor.py`

Open:
`http://127.0.0.1:5000`

### Automatic Restart / Startup
The provided `supervisor.py` restarts the Flask process if it exits.
A Linux systemd example is also included with `Restart=always`.
Students must document the method they actually use.

### Sensor / Input Source
`sensor_replay.csv` is a virtual temperature-sensor replay.

### Expected Sampling Interval
2 seconds.

### Stale-Data Definition
HEALTHY = valid/current data.
DEGRADED = missing/rejected input or more than 10 seconds since the last valid reading.
FAILED = more than 30 seconds since the last valid reading.
These are classroom thresholds.

### Threshold Rule
Trigger when accepted temperature is at or above 30.0 C.

### Alert-Control Method
Persistence: 3 consecutive valid high readings.
Hysteresis: alert clears only at or below 28.5 C.

### System-Health States
HEALTHY
DEGRADED
FAILED

A production implementation may add RECOVERING.

### Log Location
`module7_app.jsonl`
and exported `required_evidence/application_logs.csv`.

### Known Limitations
Colab is a teaching/demo environment, not a permanent unattended host.
Sensor data are simulated.
Flask test-client verification does not replace the student's own real browser screenshot.
The supervisor test proves process restart behavior, not OS-level boot behavior.

### Automatic Recovery
The demo automatically recovers from a controlled application process crash by restarting the process.

### Human Intervention Still Required
Examples:
- persistent database corruption;
- host/server outage;
- repeated crash loops;
- unresolved sensor-source failure;
- disk exhaustion;
- permission/security problems;
- invalid configuration.

### Required Evidence
See `required_evidence/` for:
dashboard running, current reading, historical view, alert active, alert cleared,
missing/degraded evidence, logs, restart/recovery, and final healthy state.

### GitHub Repository
Students must place the final project and evidence in their GitHub repository and submit the repository link.

### Course Rule
Students must be able to explain, modify, and defend all submitted code.
'''

(EXPORT/"README.md").write_text(README_TEXT,encoding="utf-8")

AI_LOG = '''# AI_LOG.md

## Tool / Model
Tool:
Model:

## Prompt(s)

## What AI Returned

## What I Changed

## How I Verified It

## What I Rejected

AI output is not operational evidence.
Verify claims with tests, logs, stored data, or observed system behavior.
'''
(EXPORT/"AI_LOG.md").write_text(AI_LOG,encoding="utf-8")

GITHUB = '''# GitHub Submission Checklist

- [ ] Repository accessible to instructor
- [ ] Flask source code included
- [ ] SQLite database or reproducible database-generation process included
- [ ] Alert logic included
- [ ] Logging configuration included
- [ ] Startup/restart configuration included
- [ ] Required evidence folder included
- [ ] README.md complete
- [ ] AI_LOG.md included if AI was used
- [ ] Incremental commit history present
- [ ] Repository link submitted to LMS
'''
(EXPORT/"GITHUB_SUBMISSION_CHECKLIST.md").write_text(GITHUB,encoding="utf-8")

shutil.copy2(DATA_FILE,EXPORT/"sensor_replay.csv")
shutil.copy2(DB_FILE,EXPORT/"module7_monitor.db")
shutil.copy2(LOG_FILE,EXPORT/"module7_app.jsonl")
shutil.copytree(APP_DIR,EXPORT/"flask_project",dirs_exist_ok=True)

print("✅ README, AI log, GitHub checklist, source, DB, and logs exported.")


In [ ]:
MANIFEST = '''EGN 321 MODULE 7 ASSIGNMENT 7.1 — SUBMISSION MANIFEST

REQUIRED EVIDENCE
1. Dashboard running
   required_evidence/dashboard_running.png
   required_evidence/dashboard_running.json
   required_evidence/dashboard_snapshot.html

2. Current reading
   required_evidence/current_reading.csv

3. Historical view
   required_evidence/historical_view.csv
   required_evidence/historical_view.png

4. Alert active
   required_evidence/alert_active_rows.csv
   required_evidence/alert_active.png

5. Alert cleared
   required_evidence/alert_cleared_row.csv
   required_evidence/alert_cleared.png

6. Missing/degraded condition
   required_evidence/missing_or_degraded_condition.csv
   required_evidence/missing_or_degraded.png

7. Logs
   required_evidence/application_logs.csv
   required_evidence/application_logs.jsonl

8. Restart/recovery
   required_evidence/restart_recovery.csv
   required_evidence/restart_recovery.png

9. Final healthy state
   required_evidence/final_healthy_state.json
   required_evidence/final_healthy_state.png

DELIVERABLES
- Flask source: flask_project/
- SQLite database: module7_monitor.db
- Alert logic: flask_project/alert_logic.py
- Logging config: flask_project/logging_config.py
- Startup/restart config: flask_project/supervisor.py
- Linux startup example: flask_project/module7-monitor.service.example
- README: README.md
- AI log: AI_LOG.md
- GitHub checklist: GITHUB_SUBMISSION_CHECKLIST.md
'''

(EXPORT/"SUBMISSION_MANIFEST.txt").write_text(MANIFEST,encoding="utf-8")
print(MANIFEST)


## Final Compliance Check


In [ ]:
required_files = [
    "required_evidence/dashboard_running.png",
    "required_evidence/current_reading.csv",
    "required_evidence/historical_view.png",
    "required_evidence/alert_active.png",
    "required_evidence/alert_cleared.png",
    "required_evidence/missing_or_degraded.png",
    "required_evidence/application_logs.jsonl",
    "required_evidence/restart_recovery.png",
    "required_evidence/final_healthy_state.png",
    "flask_project/app.py",
    "flask_project/alert_logic.py",
    "flask_project/logging_config.py",
    "flask_project/supervisor.py",
    "module7_monitor.db",
    "module7_app.jsonl",
    "README.md",
    "AI_LOG.md",
    "GITHUB_SUBMISSION_CHECKLIST.md",
    "SUBMISSION_MANIFEST.txt"
]

missing = [name for name in required_files if not (EXPORT/name).exists()]

if missing:
    print("Missing:")
    for name in missing:
        print(" -",name)
    raise RuntimeError("Assignment 7.1 package is incomplete.")

print("✅ All required Assignment 7.1 demo outputs are present.")


## Build the Complete Submission ZIP


In [ ]:
FINAL_ZIP = Path("/content/EGN321_Module7_Assignment7_1_COMPLETE_DEMO_SUBMISSION.zip")

with zipfile.ZipFile(FINAL_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in EXPORT.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(EXPORT)))

print("Created:",FINAL_ZIP)

print("\nPackage contents:")
for p in sorted(EXPORT.rglob("*")):
    if p.is_file():
        print(" -",p.relative_to(EXPORT))

# Optional download:
# from google.colab import files
# files.download(str(FINAL_ZIP))


# Demo Complete

The notebook now demonstrates and exports every required Assignment 7.1 evidence category and deliverable.

For the student's own submission, they should replace demo evidence with evidence from their own system and provide their own GitHub repository link.
